# Kaggle Runner: Experiment 1

This notebook is prepared for Kaggle GPU execution. It clones the repo into the Kaggle working directory, installs the project, runs the first experiment end to end, and exports the generated artifacts as a zip you can download back into the local repo.

Before running:
- Enable Internet in the Kaggle notebook settings.
- Use a GPU notebook.
- Set `REPO_REF` to the branch or commit you want Kaggle to execute.
- If you use `meta-llama/Llama-3.1-8B-Instruct`, add a Kaggle secret named `HF_TOKEN` and make sure the model license has been accepted on Hugging Face.

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/aaliyan1230/dual-direction-mech-interp.git"
REPO_REF = "main"  # branch or commit to run on Kaggle
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "dual-direction-mech-interp"

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
PROMPT_LIMIT = 200
EVAL_PROMPTS = 100
LOAD_IN_4BIT = True

RUN_CROSS_ABLATION = False
RUN_QUANTIZATION = False
RUN_CROSS_MODEL = False
GENERATE_FIGURES = True

HF_TOKEN_SECRET_NAME = "HF_TOKEN"
EXPORT_ARCHIVE = WORKDIR / "ddmi_kaggle_export.zip"

In [2]:
def run(cmd, cwd=None, env=None):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {printable}")
    subprocess.run([str(part) for part in cmd], cwd=cwd, env=env, check=True)


def get_hf_token(secret_name: str) -> str | None:
    token = os.environ.get(secret_name)
    if token:
        return token

    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return None

    try:
        return UserSecretsClient().get_secret(secret_name)
    except Exception:
        return None


def maybe_login_to_huggingface(model_id: str):
    token = get_hf_token(HF_TOKEN_SECRET_NAME)
    needs_token = "meta-llama" in model_id.lower()
    if needs_token and not token:
        raise RuntimeError(
            f"Set the {HF_TOKEN_SECRET_NAME} Kaggle secret before running gated model downloads."
        )

    if token:
        from huggingface_hub import login

        login(token=token, add_to_git_credential=False)
        print("Logged in to Hugging Face Hub.")
    else:
        print("No Hugging Face token configured; proceeding without hub login.")


def model_slug(model_id: str) -> str:
    return model_id.replace("/", "_").replace("-", "_")

In [3]:
if REPO_DIR.exists():
    run(["rm", "-rf", REPO_DIR])

run(["git", "clone", REPO_URL, REPO_DIR])
if REPO_REF:
    run(["git", "checkout", REPO_REF], cwd=REPO_DIR)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", ".[plot]"], cwd=REPO_DIR)
if RUN_QUANTIZATION:
    run([sys.executable, "-m", "pip", "install", "-e", ".[quant]"], cwd=REPO_DIR)

maybe_login_to_huggingface(MODEL_ID)

run([
    sys.executable,
    "-c",
    "import torch; print({'cuda_available': torch.cuda.is_available(), 'device_count': torch.cuda.device_count()})",
])

$ git clone https://github.com/aaliyan1230/dual-direction-mech-interp.git /kaggle/working/dual-direction-mech-interp


Cloning into '/kaggle/working/dual-direction-mech-interp'...


$ git checkout main
Your branch is up to date with 'origin/main'.
$ /usr/bin/python3 -m pip install --upgrade pip


Already on 'main'


$ /usr/bin/python3 -m pip install -e '.[plot]'
Obtaining file:///kaggle/working/dual-direction-mech-interp
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for ddmi (pyproject.toml): started
  Building editable for ddmi (pyproject.toml): finished with status 'done'
  Created wheel for ddmi: filename=ddmi-0.1.0-py3-none-any.whl size=3039 sha256=1510dcd5c51bd777827212201bcbf28a83a9d90bc4a3bd4a81fdbbb14f954bdc
  Stored in dire

In [4]:
safe_name = model_slug(MODEL_ID)
direction_dir = REPO_DIR / "artifacts" / "directions"
safety_artifact = direction_dir / f"{safe_name}_safety.json"
epistemic_artifact = direction_dir / f"{safe_name}_epistemic.json"
comparison_artifact = direction_dir / f"{safe_name}_comparison.json"

common_flags = ["--model-id", MODEL_ID, "--prompt-limit", str(PROMPT_LIMIT)]
if LOAD_IN_4BIT:
    common_flags.append("--load-in-4bit")

run([
    sys.executable,
    "scripts/extract_directions.py",
    *common_flags,
    "--direction-type",
    "safety",
    "--output",
    str(safety_artifact),
], cwd=REPO_DIR)

run([
    sys.executable,
    "scripts/extract_directions.py",
    *common_flags,
    "--direction-type",
    "epistemic",
    "--output",
    str(epistemic_artifact),
], cwd=REPO_DIR)

run([
    sys.executable,
    "scripts/compare_directions.py",
    "--safety-artifact",
    str(safety_artifact),
    "--epistemic-artifact",
    str(epistemic_artifact),
    "--output",
    str(comparison_artifact),
], cwd=REPO_DIR)

print({
    "safety_artifact": str(safety_artifact),
    "epistemic_artifact": str(epistemic_artifact),
    "comparison_artifact": str(comparison_artifact),
})

$ /usr/bin/python3 scripts/extract_directions.py --model-id meta-llama/Llama-3.1-8B-Instruct --prompt-limit 200 --load-in-4bit --direction-type safety --output /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json


2026-04-24 07:55:19,164 | INFO | extract_directions | Loading model: meta-llama/Llama-3.1-8B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:10<00:00, 28.19it/s, Materializing param=model.norm.weight]                              
2026-04-24 07:55:42,189 | INFO | extract_directions | Model loaded in 23.0s
2026-04-24 07:55:42,191 | INFO | extract_directions | Discovered 32 residual stream layers
2026-04-24 07:55:45,930 | INFO | extract_directions | Group A (harmful): 100 prompts, Group B (benign): 100 prompts
2026-04-24 07:55:45,931 | INFO | extract_directions | Collecting activations for group A (harmful)...
2026-04-24 07:56:09,822 | INFO | extract_directions | Group A activations collected in 23.9s
2026-04-24 07:56:09,822 | INFO | extract_directions | Collecting activations for group B (benign)...
2026-04-24 07:56:34,129 | INFO | extract_directions | Group B activations collected in 24.3s
2026-04-24 07:56:34,370 | INFO | extract

$ /usr/bin/python3 scripts/extract_directions.py --model-id meta-llama/Llama-3.1-8B-Instruct --prompt-limit 200 --load-in-4bit --direction-type epistemic --output /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json


2026-04-24 07:56:45,991 | INFO | extract_directions | Loading model: meta-llama/Llama-3.1-8B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:10<00:00, 28.36it/s, Materializing param=model.norm.weight]                              
2026-04-24 07:57:08,863 | INFO | extract_directions | Model loaded in 22.9s
2026-04-24 07:57:08,865 | INFO | extract_directions | Discovered 32 residual stream layers
2026-04-24 07:57:15,743 | INFO | extract_directions | Group A (unanswerable): 100 prompts, Group B (answerable): 100 prompts
2026-04-24 07:57:15,743 | INFO | extract_directions | Collecting activations for group A (unanswerable)...
2026-04-24 07:58:00,692 | INFO | extract_directions | Group A activations collected in 44.9s
2026-04-24 07:58:00,692 | INFO | extract_directions | Collecting activations for group B (answerable)...
2026-04-24 07:58:25,141 | INFO | extract_directions | Group B activations collected in 24.4s
2026-04-24 07:58:25,36

$ /usr/bin/python3 scripts/compare_directions.py --safety-artifact /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json --epistemic-artifact /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json --output /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_comparison.json
{'safety_artifact': '/kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json', 'epistemic_artifact': '/kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json', 'comparison_artifact': '/kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_comparison.json'}


2026-04-24 07:58:35,010 | INFO | compare_directions | Comparing 32 common layers
2026-04-24 07:58:35,012 | INFO | compare_directions |   model.layers.0: cos=-0.1850, angle=100.7°, sep_s=0.0811, sep_e=0.2574
2026-04-24 07:58:35,013 | INFO | compare_directions |   model.layers.1: cos=-0.0716, angle=94.1°, sep_s=0.1628, sep_e=0.3536
2026-04-24 07:58:35,014 | INFO | compare_directions |   model.layers.2: cos=0.0024, angle=89.9°, sep_s=0.1926, sep_e=0.4381
2026-04-24 07:58:35,016 | INFO | compare_directions |   model.layers.3: cos=0.0511, angle=87.1°, sep_s=0.2328, sep_e=0.4836
2026-04-24 07:58:35,017 | INFO | compare_directions |   model.layers.4: cos=0.0735, angle=85.8°, sep_s=0.2801, sep_e=0.5677
2026-04-24 07:58:35,018 | INFO | compare_directions |   model.layers.5: cos=0.0023, angle=89.9°, sep_s=0.3048, sep_e=0.7270
2026-04-24 07:58:35,019 | INFO | compare_directions |   model.layers.6: cos=0.0136, angle=89.2°, sep_s=0.3374, sep_e=0.8590
2026-04-24 07:58:35,021 | INFO | compare_directi

In [ ]:
if RUN_CROSS_ABLATION:
    ablation_cmd = [
        sys.executable,
        "scripts/cross_ablation.py",
        "--model-id",
        MODEL_ID,
        "--safety-direction",
        str(safety_artifact),
        "--epistemic-direction",
        str(epistemic_artifact),
        "--output",
        str(REPO_DIR / "artifacts" / "cross_ablation" / f"{safe_name}_results.json"),
        "--eval-prompts",
        str(EVAL_PROMPTS),
    ]
    if LOAD_IN_4BIT:
        ablation_cmd.append("--load-in-4bit")
    run(ablation_cmd, cwd=REPO_DIR)

if RUN_QUANTIZATION:
    import json

    quantization_target_layer = json.loads(safety_artifact.read_text())["ranked_layers"][0]["name"]
    run([
        sys.executable,
        "scripts/quantization_sweep.py",
        "--model-id",
        MODEL_ID,
        "--target-layer",
        quantization_target_layer,
        "--output",
        str(REPO_DIR / "artifacts" / "quantization" / f"{safe_name}_sweep.json"),
        "--prompt-limit",
        str(min(PROMPT_LIMIT, 100)),
    ], cwd=REPO_DIR)

if RUN_CROSS_MODEL:
    cross_model_cmd = [
        sys.executable,
        "scripts/cross_model_replication.py",
        "--output-dir",
        str(REPO_DIR / "artifacts" / "cross_model"),
        "--prompt-limit",
        str(PROMPT_LIMIT),
    ]
    if LOAD_IN_4BIT:
        cross_model_cmd.append("--load-in-4bit")
    run(cross_model_cmd, cwd=REPO_DIR)

if GENERATE_FIGURES:
    run([
        sys.executable,
        "scripts/generate_figures.py",
        "--artifacts-dir",
        "artifacts",
        "--output-dir",
        "artifacts/figures",
    ], cwd=REPO_DIR)

$ /usr/bin/python3 scripts/generate_figures.py --artifacts-dir artifacts --output-dir artifacts/figures


2026-04-24 06:24:52,541 | INFO | figures | Generating Figure 1 from artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_comparison.json
2026-04-24 06:24:53,606 | INFO | figures | Saved artifacts/figures/fig1_direction_comparison.pdf
2026-04-24 06:24:54,039 | INFO | figures | Done. Figures saved to artifacts/figures


In [8]:
run([
    sys.executable,
    "scripts/export_artifacts_bundle.py",
    "--output",
    str(EXPORT_ARCHIVE),
    "--label",
    safe_name,
    "--include-path",
    "README.md",
], cwd=REPO_DIR)

print(f"Download this file from the Kaggle Output pane: {EXPORT_ARCHIVE}")
for path in sorted((REPO_DIR / "artifacts").rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO_DIR))

$ /usr/bin/python3 scripts/export_artifacts_bundle.py --output /kaggle/working/ddmi_kaggle_export.zip --label meta_llama_Llama_3.1_8B_Instruct --include-path README.md
Wrote 12 files to /kaggle/working/ddmi_kaggle_export.zip
Download this file from the Kaggle Output pane: /kaggle/working/ddmi_kaggle_export.zip
artifacts/cross_ablation/meta_llama_Llama_3.1_8B_Instruct_results.json
artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_comparison.json
artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json
artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json
artifacts/figures/fig1_direction_comparison.pdf
artifacts/figures/fig1_direction_comparison.png
artifacts/figures/fig2_cross_ablation.pdf
artifacts/quantization/meta_llama_Llama_3.1_8B_Instruct_sweep.json


## What This Produces

By default this notebook runs the first headline experiment only:
- Safety direction extraction
- Epistemic direction extraction
- Direction comparison
- Figure generation when possible
- Export of bundled outputs to `/kaggle/working/ddmi_kaggle_export.zip`

Once Kaggle finishes, download that zip from the Output pane and extract it into the local repo before reviewing and committing artifacts.

In [7]:
import json

comparison = json.loads(comparison_artifact.read_text())
summary = comparison["summary"]

print({
    "max_cosine": round(summary["max_cosine"], 4),
    "max_cosine_layer": summary["max_cosine_layer"],
    "min_cosine": round(summary["min_cosine"], 4),
    "min_cosine_layer": summary["min_cosine_layer"],
    "mean_cosine": round(summary["mean_cosine"], 4),
    "layers_both_high_sep": summary["layers_both_high_sep"],
})

{'max_cosine': 0.1834, 'max_cosine_layer': 'model.layers.20', 'min_cosine': -0.185, 'min_cosine_layer': 'model.layers.0', 'mean_cosine': 0.0492, 'layers_both_high_sep': 31}


In [10]:
run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)

src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

run([sys.executable, "-m", "pip", "install", "-e", ".[plot]"], cwd=REPO_DIR)
run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)

$ git pull --ff-only origin main


From https://github.com/aaliyan1230/dual-direction-mech-interp
 * branch            main       -> FETCH_HEAD
   0c754eb..4386082  main       -> origin/main


Updating 0c754eb..4386082
Fast-forward
 scripts/cross_ablation.py      | 44 ++++++++++++++---------------
 src/ddmi/editing/apply_edit.py | 64 ++++++++++++++++++++++++++++++++++++++++++
 tests/test_apply_edit.py       | 33 ++++++++++++++++++++++
 3 files changed, 118 insertions(+), 23 deletions(-)
 create mode 100644 tests/test_apply_edit.py
$ /usr/bin/python3 -m pip install -e '.[plot]'
Obtaining file:///kaggle/working/dual-direction-mech-interp
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable me

In [11]:
cross_ablation_artifact = REPO_DIR / "artifacts" / "cross_ablation" / f"{safe_name}_results.json"

ablation_cmd = [
    sys.executable,
    "scripts/cross_ablation.py",
    "--model-id",
    MODEL_ID,
    "--safety-direction",
    str(safety_artifact),
    "--epistemic-direction",
    str(epistemic_artifact),
    "--output",
    str(cross_ablation_artifact),
    "--eval-prompts",
    str(EVAL_PROMPTS),
]
if LOAD_IN_4BIT:
    ablation_cmd.append("--load-in-4bit")

run(ablation_cmd, cwd=REPO_DIR)

run([
    sys.executable,
    "scripts/generate_figures.py",
    "--artifacts-dir",
    "artifacts",
    "--output-dir",
    "artifacts/figures",
], cwd=REPO_DIR)

run([
    sys.executable,
    "scripts/export_artifacts_bundle.py",
    "--output",
    str(EXPORT_ARCHIVE),
    "--label",
    f"{safe_name}_with_cross_ablation",
], cwd=REPO_DIR)

print({
    "cross_ablation_artifact": str(cross_ablation_artifact),
    "export_archive": str(EXPORT_ARCHIVE),
})

$ /usr/bin/python3 scripts/cross_ablation.py --model-id meta-llama/Llama-3.1-8B-Instruct --safety-direction /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json --epistemic-direction /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json --output /kaggle/working/dual-direction-mech-interp/artifacts/cross_ablation/meta_llama_Llama_3.1_8B_Instruct_results.json --eval-prompts 100 --load-in-4bit


2026-04-24 06:45:05,208 | INFO | cross_ablation | Safety direction: model.layers.22 (sep=0.7407)
2026-04-24 06:45:05,208 | INFO | cross_ablation | Epistemic direction: model.layers.9 (sep=1.0413)
2026-04-24 06:45:05,208 | INFO | cross_ablation | Loading model: meta-llama/Llama-3.1-8B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:10<00:00, 28.60it/s, Materializing param=model.norm.weight]                              
2026-04-24 06:45:36,455 | INFO | cross_ablation | Eval prompts: 50 safety, 50 epistemic
2026-04-24 06:45:36,456 | INFO | cross_ablation | Safety target layers: model.layers.22
2026-04-24 06:45:36,456 | INFO | cross_ablation | Epistemic target layers: model.layers.9
2026-04-24 06:45:36,456 | INFO | cross_ablation | 
=== Condition 1: BASELINE (no ablation) ===
2026-04-24 06:45:36,456 | INFO | cross_ablation |   Generating on 50 safety prompts...
The following generation flags are not valid and may be ignored: ['top_p

$ /usr/bin/python3 scripts/generate_figures.py --artifacts-dir artifacts --output-dir artifacts/figures


2026-04-24 06:55:10,706 | INFO | figures | Generating Figure 1 from artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_comparison.json
2026-04-24 06:55:11,112 | INFO | figures | Saved artifacts/figures/fig1_direction_comparison.pdf
2026-04-24 06:55:11,538 | INFO | figures | Generating Figure 2 from artifacts/cross_ablation/meta_llama_Llama_3.1_8B_Instruct_results.json
2026-04-24 06:55:11,708 | INFO | figures | Saved artifacts/figures/fig2_cross_ablation.pdf
2026-04-24 06:55:11,709 | INFO | figures | Done. Figures saved to artifacts/figures


$ /usr/bin/python3 scripts/export_artifacts_bundle.py --output /kaggle/working/ddmi_kaggle_export.zip --label meta_llama_Llama_3.1_8B_Instruct_with_cross_ablation
Wrote 10 files to /kaggle/working/ddmi_kaggle_export.zip
{'cross_ablation_artifact': '/kaggle/working/dual-direction-mech-interp/artifacts/cross_ablation/meta_llama_Llama_3.1_8B_Instruct_results.json', 'export_archive': '/kaggle/working/ddmi_kaggle_export.zip'}


In [13]:
import json

cross_results = json.loads(cross_ablation_artifact.read_text())
rows = {row["condition"]: row for row in cross_results["results"]}
summary = cross_results["summary"]

print({
    "target_layers": cross_results["config"]["target_layers"],
    "baseline": {
        "safety_refusal_rate": round(rows["baseline"]["safety_refusal_rate"], 3),
        "epistemic_abstention_rate": round(rows["baseline"]["epistemic_abstention_rate"], 3),
    },
    "ablate_safety": {
        "safety_refusal_rate": round(rows["ablate_safety"]["safety_refusal_rate"], 3),
        "epistemic_abstention_rate": round(rows["ablate_safety"]["epistemic_abstention_rate"], 3),
    },
    "ablate_epistemic": {
        "safety_refusal_rate": round(rows["ablate_epistemic"]["safety_refusal_rate"], 3),
        "epistemic_abstention_rate": round(rows["ablate_epistemic"]["epistemic_abstention_rate"], 3),
    },
    "cross_contamination": {
        "safety_to_epistemic": round(summary["cross_contamination_safety_to_epistemic"], 3),
        "epistemic_to_safety": round(summary["cross_contamination_epistemic_to_safety"], 3),
    },
})

{'target_layers': {'epistemic': ['model.layers.9'], 'safety': ['model.layers.22']}, 'baseline': {'safety_refusal_rate': 0.98, 'epistemic_abstention_rate': 0.82}, 'ablate_safety': {'safety_refusal_rate': 0.96, 'epistemic_abstention_rate': 0.88}, 'ablate_epistemic': {'safety_refusal_rate': 0.98, 'epistemic_abstention_rate': 0.82}, 'cross_contamination': {'safety_to_epistemic': 0.06, 'epistemic_to_safety': 0.0}}


In [5]:
import json

quantization_artifact = REPO_DIR / "artifacts" / "quantization" / f"{safe_name}_sweep.json"
quantization_target_layer = json.loads(safety_artifact.read_text())["ranked_layers"][0]["name"]
quantization_prompt_limit = min(PROMPT_LIMIT, 100)

quantization_cmd = [
    sys.executable,
    "scripts/quantization_sweep.py",
    "--model-id",
    MODEL_ID,
    "--fp16-safety-artifact",
    str(safety_artifact),
    "--fp16-epistemic-artifact",
    str(epistemic_artifact),
    "--target-layer",
    quantization_target_layer,
    "--output",
    str(quantization_artifact),
    "--prompt-limit",
    str(quantization_prompt_limit),
]

print({
    "recommended_stage": "quantization_sweep",
    "target_layer": quantization_target_layer,
    "prompt_limit": quantization_prompt_limit,
    "output": str(quantization_artifact),
    "command": " ".join(shlex.quote(str(part)) for part in quantization_cmd),
})

{'recommended_stage': 'quantization_sweep', 'target_layer': 'model.layers.22', 'prompt_limit': 100, 'output': '/kaggle/working/dual-direction-mech-interp/artifacts/quantization/meta_llama_Llama_3.1_8B_Instruct_sweep.json', 'command': '/usr/bin/python3 scripts/quantization_sweep.py --model-id meta-llama/Llama-3.1-8B-Instruct --fp16-safety-artifact /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json --fp16-epistemic-artifact /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json --target-layer model.layers.22 --output /kaggle/working/dual-direction-mech-interp/artifacts/quantization/meta_llama_Llama_3.1_8B_Instruct_sweep.json --prompt-limit 100'}


In [6]:
import json

print("Launching quantization sweep...")
run(quantization_cmd, cwd=REPO_DIR)

if not quantization_artifact.exists():
    raise FileNotFoundError(quantization_artifact)

quant_results = json.loads(quantization_artifact.read_text())
drift_table = quant_results.get("drift_table", [])
print(
    json.dumps(
        {
            "output": str(quantization_artifact),
            "target_layer": quant_results.get("config", {}).get("target_layer"),
            "num_conditions": len(drift_table),
            "drift_table": drift_table,
        },
        indent=2,
    )
)

Launching quantization sweep...
$ /usr/bin/python3 scripts/quantization_sweep.py --model-id meta-llama/Llama-3.1-8B-Instruct --fp16-safety-artifact /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_safety.json --fp16-epistemic-artifact /kaggle/working/dual-direction-mech-interp/artifacts/directions/meta_llama_Llama_3.1_8B_Instruct_epistemic.json --target-layer model.layers.22 --output /kaggle/working/dual-direction-mech-interp/artifacts/quantization/meta_llama_Llama_3.1_8B_Instruct_sweep.json --prompt-limit 100


2026-04-24 07:58:57,213 | INFO | quantization_sweep | Using provided FP16 artifacts as baseline for layer model.layers.22
2026-04-24 07:58:57,213 | INFO | quantization_sweep | 
=== Precision: bnb_nf4 ===
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:10<00:00, 28.38it/s, Materializing param=model.norm.weight]                              
2026-04-24 07:59:19,439 | INFO | quantization_sweep | Model loaded in 22.2s
2026-04-24 08:00:15,250 | INFO | quantization_sweep | Directions extracted in 55.8s
2026-04-24 08:00:15,593 | INFO | quantization_sweep | 
=== Precision: bnb_int8 ===
Loading weights: 100%|██████████| 291/291 [00:22<00:00, 12.72it/s, Materializing param=model.norm.weight]                              
2026-04-24 08:00:41,217 | INFO | quantization_sweep | Model loaded in 25.6s
2026-04-24 08:01:34,746 | INFO | quantization_sweep | Directions extracted in 53.5s
2026-04-24 08:01:35,108 | INFO | quantization_sweep | fp16: safety_cos=

{
  "output": "/kaggle/working/dual-direction-mech-interp/artifacts/quantization/meta_llama_Llama_3.1_8B_Instruct_sweep.json",
  "target_layer": null,
  "num_conditions": 3,
  "drift_table": [
    {
      "epistemic_cosine_vs_fp16": 1.0,
      "epistemic_norm_ratio": 1.0,
      "epistemic_separability": null,
      "precision": "fp16",
      "safety_cosine_vs_fp16": 1.0,
      "safety_norm_ratio": 1.0,
      "safety_separability": null
    },
    {
      "epistemic_cosine_vs_fp16": 0.977554740363933,
      "epistemic_norm_ratio": 1.0000000000000002,
      "epistemic_separability": 0.48698970129851826,
      "precision": "bnb_nf4",
      "safety_cosine_vs_fp16": 0.9940238375404946,
      "safety_norm_ratio": 1.0,
      "safety_separability": 0.7604332676808384
    },
    {
      "epistemic_cosine_vs_fp16": 0.9349445736576494,
      "epistemic_norm_ratio": 1.0000000000000002,
      "epistemic_separability": 0.4439153815922095,
      "precision": "bnb_int8",
      "safety_cosine_vs_fp16":

In [7]:
import json

if not quantization_artifact.exists():
    raise FileNotFoundError(quantization_artifact)

quant_results = json.loads(quantization_artifact.read_text())
print(json.dumps({
    "artifact": str(quantization_artifact),
    "target_layer": quant_results.get("target_layer"),
    "precisions": quant_results.get("precisions", []),
    "drift_table": quant_results.get("drift_table", []),
}, indent=2))

{
  "artifact": "/kaggle/working/dual-direction-mech-interp/artifacts/quantization/meta_llama_Llama_3.1_8B_Instruct_sweep.json",
  "target_layer": "model.layers.22",
  "precisions": [
    "fp16",
    "bnb_nf4",
    "bnb_int8"
  ],
  "drift_table": [
    {
      "epistemic_cosine_vs_fp16": 1.0,
      "epistemic_norm_ratio": 1.0,
      "epistemic_separability": null,
      "precision": "fp16",
      "safety_cosine_vs_fp16": 1.0,
      "safety_norm_ratio": 1.0,
      "safety_separability": null
    },
    {
      "epistemic_cosine_vs_fp16": 0.977554740363933,
      "epistemic_norm_ratio": 1.0000000000000002,
      "epistemic_separability": 0.48698970129851826,
      "precision": "bnb_nf4",
      "safety_cosine_vs_fp16": 0.9940238375404946,
      "safety_norm_ratio": 1.0,
      "safety_separability": 0.7604332676808384
    },
    {
      "epistemic_cosine_vs_fp16": 0.9349445736576494,
      "epistemic_norm_ratio": 1.0000000000000002,
      "epistemic_separability": 0.4439153815922095,
    